# HECKTOR preprocessing preview (Google Colab)

Run the RADCURE-style preprocessing pipeline on a HECKTOR case step-by-step:
1. Get the case
2. Run TotalSegmentator and visualize
3. Generate background (head/anatomical mask)
4. Generate combined mask
5. Add tumor mask (GTVp + GTVn as single label)
5b. Generate PDF (CT | CT + organs + tumor with legend)
6. Save results for model prediction

HECKTOR: `{case_id}.nii.gz` = mask (0=bg, 1=GTVp, 2=GTVn), `{case_id}__CT.nii.gz` = CT. Shape (512, 512, z).

## Colab setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (replace with your repo URL if needed)
!git clone https://github.com/YOUR_ORG/radcure-medical-imaging.git
%cd radcure-medical-imaging

In [ ]:
!pip install -r requirements.txt
!pip install totalsegmentator

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

if '/content/radcure-medical-imaging' not in sys.path:
    sys.path.insert(0, '/content/radcure-medical-imaging')

from radcure_processor.io.nifti_handler import NIfTIHandler
from radcure_processor.core.mask_generator import MaskGenerator
from radcure_processor.utils.organ_dictionary import OrganDictionary
from radcure_processor.core.segmentator import TotalSegmentatorWrapper

# Config
CASES_ROOT = "/content/drive/MyDrive/phD/phD-Petia/Trainings/Hecktor/cases"
case_id = "CHUM-001"
case_folder = os.path.join(CASES_ROOT, case_id)
path_ct = os.path.join(case_folder, f"{case_id}__CT.nii.gz")
path_mask = os.path.join(case_folder, f"{case_id}.nii.gz")
slice_expansion = 5

print("Case folder:", case_folder)
print("CT:", path_ct)
print("Mask:", path_mask)

## Step 1 — Get the case

In [ ]:
ct_nii = nib.load(path_ct)
mask_nii = nib.load(path_mask)
ct_vol = ct_nii.get_fdata().astype(np.float32)
mask_vol = mask_nii.get_fdata().astype(np.int32)

print("CT shape:", ct_vol.shape)
print("Mask shape:", mask_vol.shape)
print("Mask unique labels:", np.unique(mask_vol))

# Slice range: indices where mask has 1 or 2, then expand
z = mask_vol.shape[2]
non_zero_slices = np.where(np.any(mask_vol != 0, axis=(0, 1)))[0]
if len(non_zero_slices) == 0:
    slices_to_use = list(range(z))
else:
    start = max(int(non_zero_slices.min()) - slice_expansion, 0)
    end = min(int(non_zero_slices.max()) + slice_expansion, z - 1)
    slices_to_use = list(range(start, end + 1))
print("Slices to use:", len(slices_to_use), "(indices", slices_to_use[0], "..", slices_to_use[-1], ")")

In [ ]:
# Visualize one axial slice: CT + mask overlay (GTVp=1, GTVn=2)
sl = len(slices_to_use) // 2
idx = slices_to_use[sl]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
p1, p99 = np.percentile(ct_vol, (1, 99))
ct_slice = np.clip((ct_vol[:, :, idx] - p1) / (p99 - p1 + 1e-8), 0, 1)
axes[0].imshow(ct_slice.T, cmap="gray", origin="lower")
axes[0].set_title(f"CT slice {idx}")
axes[0].axis("off")
axes[1].imshow(ct_slice.T, cmap="gray", origin="lower")
m = mask_vol[:, :, idx]
axes[1].imshow(np.ma.masked_where(m == 0, m).T, cmap="nipy_spectral", alpha=0.6, origin="lower")
axes[1].set_title(f"CT + mask (1=GTVp, 2=GTVn) slice {idx}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Step 2 — Run TotalSegmentator and visualize

In [ ]:
ts_output_path = os.path.join(case_folder, "total_segmentator_output")
tasks_to_run = [
    'head_glands_cavities', 'head_muscles', 'headneck_bones_vessels',
    'headneck_muscles', 'oculomotor_muscles', 'craniofacial_structures'
]
segmentator = TotalSegmentatorWrapper(fast=True)
ts_output_path = segmentator.run_tasks(case_id, path_ct, case_folder, tasks_to_run)
print("TotalSegmentator output:", ts_output_path)

In [ ]:
def show_ct_ts(ct_slice, norm_ts, idx, index_to_name, combined_ts_vol):
    ts_slice = combined_ts_vol[:, :, idx]
    unique_in_slice = sorted(set(np.unique(ts_slice).astype(int)) - {0})
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(ct_slice.T, cmap="gray", origin="lower")
    axes[0].set_title(f"CT (slice {idx})")
    axes[0].axis("off")
    axes[1].imshow(ct_slice.T, cmap="gray", origin="lower")
    axes[1].imshow(ts_slice.T, cmap=cmap_ts, norm=norm_ts, alpha=0.5, origin="lower")
    axes[1].set_title(f"CT + all organs (slice {idx})")
    axes[1].axis("off")
    legend_patches = [mpatches.Patch(facecolor=cmap_ts(norm_ts(l)), edgecolor="k", label=index_to_name.get(l, str(l))) for l in unique_in_slice]
    if legend_patches:
        axes[1].legend(handles=legend_patches, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Build combined all-organs label volume and show CT | CT+all organs (RADCURE-style)
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm

individual_paths = []
for d in os.listdir(ts_output_path.rstrip('/')):
    dp = os.path.join(ts_output_path, d)
    if os.path.isdir(dp):
        for f in os.listdir(dp):
            if f.endswith('.nii.gz'):
                individual_paths.append(os.path.join(dp, f))
print(f"Found {len(individual_paths)} organ mask files")

if individual_paths:
    h, w, z = ct_vol.shape
    combined_ts_vol = np.zeros((h, w, z), dtype=np.int32)
    paths_ordered = list(reversed(individual_paths))
    organ_names = [os.path.basename(p).replace('.nii.gz', '') for p in paths_ordered]
    index_to_name = {i + 1: name for i, name in enumerate(organ_names)}
    for i, op in enumerate(paths_ordered):
        ov = nib.load(op).get_fdata()
        if ov.shape != combined_ts_vol.shape:
            ov = np.transpose(ov, (1, 2, 0)) if ov.shape[0] == z else ov
        combined_ts_vol[ov == 1] = i + 1
    n_labels = max(combined_ts_vol.max(), 1)
    cmap_ts = plt.cm.get_cmap("nipy_spectral", max(n_labels + 1, 2))
    bounds = np.arange(-0.5, n_labels + 1.5, 1)
    norm_ts = BoundaryNorm(bounds, cmap_ts.N)
    show_ct_ts(ct_slice, norm_ts, idx, index_to_name, combined_ts_vol)
     

In [ ]:
## Run over slices 
for idx in range(0, ct_vol.shape[2]):
  ct_slice = np.clip((ct_vol[:, :, idx] - p1) / (p99 - p1 + 1e-8), 0, 1)
  show_ct_ts(ct_slice, norm_ts, idx, index_to_name, combined_ts_vol)


## Step 3 — Generate background (head/anatomical mask)

In [ ]:
organ_dict = OrganDictionary(None)
mask_gen = MaskGenerator(organ_dict)
background_array_int = mask_gen.generate_background_array(slices_to_use, path_ct)
print("Background array:", len(background_array_int), "slices")

In [ ]:
# Visualize: one slice — CT and anatomical (1) vs background (0)
bg_slice = background_array_int[sl]
anatomical_slice = (bg_slice == 0).astype(float)
anatomical_slice[anatomical_slice == 0] = np.nan
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(ct_slice.T, cmap="gray", origin="lower")
ax.imshow(anatomical_slice.T, cmap="Reds", alpha=0.4, origin="lower")
ax.set_title("CT + anatomical region (background removed)")
ax.axis("off")
plt.show()

## Step 4 — Generate combined mask

In [ ]:
combined_mask_array, organ_dict = mask_gen.generate_combined_mask(
    slices_to_use, background_array_int, ts_output_path.rstrip('/')
)
print("Combined mask:", len(combined_mask_array), "slices; unique labels (sample):", np.unique(combined_mask_array[sl]))

In [ ]:
# Visualize all slices 
for sl in range(0, len(slices_to_use)):
  idx = slices_to_use[sl]
  ct_slice = np.clip((ct_vol[:, :, idx] - p1) / (p99 - p1 + 1e-8), 0, 1)
  cm_slice = combined_mask_array[sl]
  fig, ax = plt.subplots(1, 1, figsize=(6, 6))
  ax.imshow(ct_slice.T, cmap="gray", origin="lower")
  cm_plot = np.ma.masked_where(cm_slice == 0, cm_slice)
  ax.imshow(cm_plot.T, cmap="nipy_spectral", alpha=0.6, origin="lower")
  ax.set_title(f'CT + combined mask (bg, anatomical_region, other-tissue, organs) {idx}')
  ax.axis("off")  
  plt.show()

## Step 5 — Add tumor mask (GTVp + GTVn as single label)

In [ ]:
tumor_value = organ_dict.add_tumor_index()
for i in range(len(combined_mask_array)):
    hecktor_slice = mask_vol[:, :, slices_to_use[i]]
    combined_mask_array[i][(hecktor_slice == 1) | (hecktor_slice == 2)] = tumor_value
combined_mask_array_tumor = combined_mask_array
print("Tumor (GTVp+GTVn combined) index:", tumor_value)

In [ ]:
# Visualize: CT + final mask with tumor
fin_slice = combined_mask_array_tumor[sl]
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(ct_slice.T, cmap="gray", origin="lower")
fin_plot = np.ma.masked_where(fin_slice == 0, fin_slice)
ax.imshow(fin_plot.T, cmap="nipy_spectral", alpha=0.6, origin="lower")
ax.set_title("CT + final mask (incl. tumor)")
ax.axis("off")
plt.show()

## Step 5b — Generate PDF (CT | CT + all organs + tumor with legend)

Optionally save a PDF with one page per slice: left = CT, right = CT + combined mask (organs + tumor) with legend.

In [ ]:
# Option: set to True to save PDF; path is under case_folder/output/
SAVE_PREVIEW_PDF = True
results_path = os.path.join(case_folder, "output")
os.makedirs(results_path, exist_ok=True)
preview_pdf_path = os.path.join(results_path, f"{case_id}_preview.pdf") if SAVE_PREVIEW_PDF else None

import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import BoundaryNorm, ListedColormap

# Index -> organ name for legend (GTVp = tumor)
index_to_organ = {v: k for k, v in organ_dict.dictionary.items()}
gtvp_index = organ_dict.get("GTVp")
max_label = max(organ_dict.dictionary.values()) if organ_dict.dictionary else 1
colormap_size = max_label + 1
base_cmap = plt.cm.get_cmap("nipy_spectral", max(colormap_size, 2))
colors = base_cmap(np.arange(colormap_size))
colors[0, :] = [0.0, 0.0, 0.0, 0.0]
if gtvp_index is not None and gtvp_index < colormap_size:
    colors[gtvp_index, :] = [1.0, 0.0, 0.0, 1.0]
cmap_mask = ListedColormap(colors)
norm_mask = BoundaryNorm(np.arange(-0.5, colormap_size + 0.5, 1), cmap_mask.N)

pdf = PdfPages(preview_pdf_path) if preview_pdf_path else None
try:
    for i in range(len(combined_mask_array_tumor)):
        idx = slices_to_use[i]
        ct_slice = np.clip((ct_vol[:, :, idx] - p1) / (p99 - p1 + 1e-8), 0, 1)
        mask_slice = combined_mask_array_tumor[i]
        unique_in_slice = sorted(set(np.unique(mask_slice).astype(int)) - {0})
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(ct_slice.T, cmap="gray", origin="lower")
        axes[0].set_title(f"CT (slice {idx})")
        axes[0].axis("off")
        axes[1].imshow(ct_slice.T, cmap="gray", origin="lower")
        axes[1].imshow(mask_slice.T, cmap=cmap_mask, norm=norm_mask, alpha=0.5, origin="lower")
        axes[1].set_title(f"CT + organs + tumor (slice {idx})")
        axes[1].axis("off")
        legend_patches = [mpatches.Patch(facecolor=colors[l], edgecolor="k", label=index_to_organ.get(l, str(l))) for l in unique_in_slice if l < len(colors)]
        if legend_patches:
            axes[1].legend(handles=legend_patches, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
        fig.suptitle(f"Slice {i+1}/{len(combined_mask_array_tumor)} (index {idx})", fontsize=12)
        plt.tight_layout()
        if pdf:
            pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
finally:
    if pdf:
        pdf.close()
        print("Saved PDF:", preview_pdf_path)
if not SAVE_PREVIEW_PDF:
    print("PDF not saved (set SAVE_PREVIEW_PDF = True to save).")

## Step 6 — Save results for model prediction

In [ ]:
ct_img_array = mask_gen.generate_ct_images(path_ct, slices_to_use)
results_path = os.path.join(case_folder, "output")
results_path_images = os.path.join(results_path, "image")
results_path_labels = os.path.join(results_path, "labels")
os.makedirs(results_path_images, exist_ok=True)
os.makedirs(results_path_labels, exist_ok=True)

image_path = NIfTIHandler.save_as_nii(ct_img_array, results_path_images, case_id, dtype_needed=False)
print("Saved image:", image_path)

# Labels: save as case_XXX.nii.gz (nnUNet convention, no _0000)
label_volume = np.stack(combined_mask_array_tumor)
volume_t = np.transpose(label_volume, (1, 2, 0))
label_path = os.path.join(results_path_labels, f"case_{case_id.split('-')[1]}.nii.gz")
nib.save(nib.Nifti1Image(volume_t.astype(np.uint8), np.eye(4)), label_path)
print("Saved labels:", label_path)

**Using saved outputs for prediction:** Copy `output/image/case_XXX_0000.nii.gz` into your nnUNet dataset `imagesTs/` and `output/labels/case_XXX.nii.gz` into `labelsTs/` (or use as training data in `imagesTr/` and `labelsTr/`). The dataset expects images with `_0000` suffix and labels without.